# Обучение классификатора (Этап 4)

Два режима:
1. **Random Forest** на морфометрических метриках (baseline)
2. **MLP** на latent vector + морфометрия (основной)

**Предусловие:** Загрузите `best_autoencoder.pt` (от этапа 1) в шаге 4.

In [ ]:
# 1. Монтируем Google Drive (для данных)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Скачиваем актуальный код
%cd /content
!rm -rf TriView3D
!git clone https://github.com/3x6dll9ff/TriView3D.git
%cd TriView3D

In [ ]:
# 3. Зависимости
!pip install -r requirements.txt

In [ ]:
# 4. Подготовка: данные + копирование базовой модели
import os, shutil

!rm -rf /content/data
!unzip -q /content/drive/MyDrive/data_processed.zip -d /content/

# Базовая модель (после этапа 1 лежит в /content/)
os.makedirs('results', exist_ok=True)
src = '/content/best_autoencoder.pt'
dst = 'results/best_autoencoder.pt'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f'Base model скопирован: {src} → {dst}')
else:
    print('ОШИБКА: /content/best_autoencoder.pt не найден!')
    print('Скачайте его по результатам этапа 1 и загрузите в Colab.')
    from google.colab import files
    print('Загрузите best_autoencoder.pt с компьютера:')
    uploaded = files.upload()
    for filename in uploaded:
        with open(os.path.join('results', filename), 'wb') as f:
            f.write(uploaded[filename])
        print(f'Сохранено: results/{filename}')

In [ ]:
# 5a. Random Forest (baseline — быстро)
!DATA_PATH=$(find /content -name "metadata.csv" | head -n 1 | xargs dirname) && \
 python3 src/train_classifier.py \
    --data_dir "$DATA_PATH" \
    --mode rf

In [ ]:
# 5b. MLP классификатор на latent + морфометрия
!DATA_PATH=$(find /content -name "metadata.csv" | head -n 1 | xargs dirname) && \
 python3 src/train_classifier.py \
    --data_dir "$DATA_PATH" \
    --mode latent \
    --autoencoder results/best_autoencoder.pt \
    --input_mode tri \
    --epochs 40 \
    --batch_size 16 \
    --lr 1e-3

In [ ]:
# 6. Скачиваем классификатор на компьютер
from google.colab import files
import os
if os.path.exists("results/best_classifier.pt"):
    files.download("results/best_classifier.pt")
else:
    print("ОШИБКА: best_classifier.pt не найден!")